# W07b Colab A ·토큰·확률·생성

생성형 인공지능 ·2026년10월14일 ·동덕여자대학교 데이터 사이언스 전공 ·유원상 교수

CPU에서 첫 셀부터 실행합니다. 두 노트북은 독립적입니다. GPU·유료API·개인파일 업로드·대형모델 다운로드가 필요 없습니다. 내장 작은 표와 합성 문자열을 사용하며 실제 LLM 출력이라고 표시하지 않습니다.

파일 → Drive에 사본 저장 후 사용하세요. 설치한 모듈을 이미 import했다면 런타임을 재시작하고 첫 셀부터 실행합니다. 실패 시 오류를 확인하며 가짜 결과로 대체하지 않습니다.

각 문제는 먼저 읽고 자신의 코드 셀에서 수정합니다. 자동 시연과 문제 답 변수를 분리했으므로 미완성 답이 있어도 시연 검사는 실행됩니다. 전체 실행 통과는 자신이 모든 문제를 풀었다는 뜻이 아닙니다.


## 공통 코드: 정의와 입력·출력

설치한 태그 `2026-fall-w07b`와 같은 정의 행입니다. 함수 이름을 클릭하여 구현을 바로 확인할 수 있습니다. 아래 API는 실습 편의를 위한 것이며, 연습문항은 기본 Python으로 풉니다.

### [softmax_scores](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L14)

유한 점수 N개 → 합1 확률 N개. T>0, 기본1; 입력은 수정하지 않는다.

최댓값을 뺀 뒤 T로 나누어 지수를 계산한다. 예: [0,log(2)] → [1/3,2/3].
빈 입력·비유한 값·양수가 아닌 T는 ValueError. 학습이나 추출은 하지 않는다.

### [filter_probs](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L31)

합1 비음수 확률 → {base, after_k, final, order, kept}.

top_k=0은 해제; 1..N 정수는 상위 개수. top_p는 (0,1], 기본1은 해제.
순서: top-k → 재정규화 → top-p → 재정규화. 경계를 넘는 토큰 포함.
동률은 원 인덱스 순서; final은 원 순서이며 제외 후보0. 입력 변경 없음.
예: filter_probs([.6,.3,.1],top_k=2)['final'] == [2/3,1/3,0].

### [attention_row](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L64)

Query1개·Key N×d·Value N×v → 점수/허용/가중치/출력 dict.

position은0-based Query 위치; causal=True이면 j<=position만 허용한다.
점수=q·k/sqrt(d), 허용점수에 softmax, 출력폭v의 Value 가중합.
학습·다운로드 없음. 잘못된 차원·비유한 값은 ValueError.
예: attention_row([1],[[0],[0]],[[2],[6]],1)['output'] == [4.0].

### [bigram_counts](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L92)

인접 문자 전이 횟수. 공백도 문자이며 끝→처음 연결 없음.

입력 길이2이상 문자열. 반환은 {출발문자:{다음문자:횟수}} 새 dict.
예: 'xyx' → {'x':{'y':1},'y':{'x':1}}. 파일을 읽지 않는다.

### [generate_one](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L107)

직전 문자 전이표에서1문자 선택. {context,next,table,mode} 반환.

seed는 로컬 Random에만 적용하여 전역 난수 상태를 바꾸지 않는다.
greedy 동률은 문자 정렬 순서. 매 호출 seed를 다시 설정하는 수업 실험이며,
긴 생성의 연속 난수 스트림을 모사한다고 주장하지 않는다. 미관측 문맥은 오류.

### [definition_link](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_core.py#L132)

설치한 수업 함수의 실제 정의 시작 행 GitHub URL; 네트워크 요청 없음.

수업 패키지 함수만 허용한다. ref는 설치한 tag/commit과 같게 지정한다.

### [probability_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_app.py#L9)

세 설정 → 표·설명·막대그래프. 가상 로짓[2.4,1.1,.2,-.6], 실제LLM 아님.

top_k 정수0..4, top_p(0,1], T>0. summary_fn(result)->str는 학생 연결 지점.
반환 (후보표 list, Markdown str, matplotlib Figure); 호출자는 Figure를 닫는다.

### [attention_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_app.py#L32)

0-based 위치0..2와 마스크 → Key표·출력벡터·가중치 그래프.

Q=K=[[1,0],[0,1],[1,1]], V=[[2,1],[0,4],[3,-1]]은 명시한 수계산 예다.
(표 list, 설명 str, Figure) 반환; 학습·다운로드 없음.

### [generation_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_app.py#L50)

문맥·정수seed·greedy → 새문맥·전이표·모드설명. 내장 합성 말뭉치 사용.

전이표 열은 다음문자/횟수/확률. 매 클릭1문자 추가. 실패는 명확한 오류로 전달.

### [build_review_app](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07b/src/luna_genai/review_app.py#L59)

3패널 Gradio Blocks 생성, 서버는 시작하지 않음. CPU·API키/모델 다운로드 없음.

summary_fn(result)->str를 연결하면 확률 패널에 학생 설명을 표시한다.
예: app=build_review_app(); app.launch(share=True). 공유 URL은 런타임에 의존.


In [ ]:
import os, sys, subprocess, math
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
IN_COLAB = 'google.colab' in sys.modules
COURSE_REF = '2026-fall-w07b'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        f'luna-genai @ git+https://github.com/lunalab-ai/genAI.git@{COURSE_REF}#subdirectory=src'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio==6.26.0', 'matplotlib>=3.8,<4'], check=True)
import luna_genai
assert luna_genai.__version__ == '0.1.12', '패키지 버전이 다릅니다. 설치 후 런타임을 재시작하세요.'
from luna_genai.review_core import softmax_scores, filter_probs, attention_row, bigram_counts, generate_one, definition_link
from luna_genai.review_app import probability_view, attention_view, generation_view, build_review_app
from IPython.display import display, Markdown
print('Python', sys.version.split()[0], '| course', luna_genai.__version__, COURSE_REF)


### 실제 소스 바로가기 재확인

아래 함수는 설치 파일에서 정의 줄을 찾아 같은 고정 태그 URL을 만듭니다. API 이름을 눌러 입력 검증과 중간 계산을 읽어 보세요.


In [ ]:
for fn in [softmax_scores, filter_probs, attention_row, bigram_counts, generate_one, probability_view, attention_view, generation_view, build_review_app, definition_link]:
    display(Markdown(f'[{fn.__name__}]({definition_link(fn, COURSE_REF)})'))


## 1. 설명 예제: 전이 횟수에서 확률로

문자열 `xyxz`의 전이는xy,yx,xz입니다. x행은y와z가각1회이므로 반씩입니다. 다음 출력에서 합과 분모를 확인합니다. 이는 문제 은행과 다른 숫자·문자 예제입니다.


In [ ]:
demo_counts = bigram_counts('xyxz')
print(demo_counts)
print(generate_one('x', demo_counts, seed=11, greedy=True))


## 2. 설명 예제: 확률과 필터

로짓[0,ln5]는지수[1,5],확률[1/6,5/6]입니다. 다른표[0.55,0.25,0.20]에top-p=0.7을 적용하면 누적0.55→0.80에서 멈춥니다. 출력dict의base/after_k/final을 서로 구별합니다.


In [ ]:
print(softmax_scores([0, math.log(5)]))
print(filter_probs([.55,.25,.20], top_p=.7))


## 3. 앱에서 쓸 callback을 먼저 호출

확률 표와 그림을 만드는 함수의 반환 값은 표,설명,Figure입니다. UI버튼도 같은 계산을 호출합니다. 이 단계는 브라우저 조작과 구별되는 함수 실행입니다.


In [ ]:
import matplotlib.pyplot as plt
table, report, figure = probability_view(1.0, 0, 0.85)
print(table); print(report); display(figure); plt.close(figure)


## 4. Python 연습

Notion·PDF와같은Q번호입니다. 빈칸/오류 코드는 설명 셀에 있으며 그대로 자동 실행하지 않습니다. 필요한 문법은 문제에 제공됩니다. 자신의 코드 셀에서 완성하고 출력뿐 아니라 이유도 기록하세요.


### Q08 ·실험 기록 읽기 (빈칸)

빈칸에 딕셔너리 키 하나를 넣어 사용한 seed를 출력하라. 딕셔너리는 키로 값을 찾는다. 출력과 seed를 기록하는 이유도 설명하라.

```python
record = {"prompt": "작은 정원", "seed": 17, "temperature": 0.7}
print(record[___])
```

목표: 입문·책임의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q08_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q13 ·전이 분모 채우기 (빈칸)

빈칸을 채워 문자 a의 다음 문자 분포를 구하라. sum(목록)은 합, items()는 (키, 값) 쌍을 차례로 준다.

```python
counts = {"b": 4, "c": 2}
total = ___
p = {k: v / total for k, v in counts.items()}
print(p)
```

목표: Colab·bigram의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q13_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q14 ·인접쌍 인덱스 오류 (디버깅)

아래 코드가 마지막 반복에서 실패한다. 반복 범위를 최소한으로 수정하고 생성되는 쌍을 순서대로 써라. range(n)은 0부터 n −1이다.

```python
text = "aabc"
pairs = []
for i in range(len(text)):
    pairs.append(text[i] + text[i+1])
print(pairs)
```

목표: Colab·bigram의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q14_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q15 ·직전 문자 읽기 (빈칸)

문자 bigram의 조건으로 현재 문맥의 마지막 문자 하나를 읽도록 빈칸을 채워라. 음 수 인덱스 −1은 마지막 원소를 뜻한다.

```python
context = "별빛달"
last = context[___]
print(last)
```

목표: Colab·bigram의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q15_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q16 ·현재 커널의 변수 (디버깅)

새 런타임에서 아래 두 셀 을 표시된 순서로 실행한다. 원인과 복구 순서를 쓰고, 정상 출력도 구하라.

```python
# 셀 A
print(total / n)
# 셀 B
n = 4
total = 10
```

목표: Colab·bigram의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q16_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q25 ·토큰 ID 목록 만들기 (빈칸)

빈칸을 채워 각 토큰의 ID를 원래 순서대로 읽어라. vocab[t]는 토큰 t의 ID를 준다. 결과의 ID 합이 의미를 나타내는지도 설명하라.

```python
vocab = {"봄": 2, "꽃": 0, "빛": 1}
tokens = ["꽃", "봄", "꽃"]
ids = [___ for t in tokens]
print(ids)
```

목표: 토큰·표현의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q25_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q26 ·벡터 덧셈 오류 (디버깅)

아래 코드는 위치 표현을 더 하려했지만 원하는 결과가 아니다. 실제 결과를 쓰고, 길이 2의 원소별 합을 반환하도록 수정하라.

```python
e = [1, 3]
p = [2, -1]
x = e + p
print(x)
```

목표: 토큰·표현의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q26_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q27 ·임베딩 조회 구현 (빈칸)

빈칸을 채워 ID마다 표의 해당 행을 읽어라. table[i]는 0-based i 행이다. 반환 목록의 모양도 써라.

```python
table = [[2, 0], [0, 3], [1, -1]]
ids = [1, 2]
x = [___ for i in ids]
print(x)
```

목표: 토큰·표현의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q27_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q28 ·반복 토큰의 초기 표현 (출력·해석)

코드 출력을 쓰고, 이 결과만으로 서로 다른 위치의 최종 문맥 표현도 같다고 말할 수 있는지 설명하라.

```python
table = [[1, 0], [0, 1]]
ids = [1, 0, 1]
x = [table[i] for i in ids]
print(x[0] == x[2])
print(len(x), len(x[0]))
```

목표: 토큰·표현의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q28_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q39 ·softmax 분모 (빈칸)

가중치 w는 로짓에 이미 지수를 적용한 값이다. 빈칸을 채워 확률을 만들고 결과를 써라.

```python
w = [2, 3, 5]
p = [v / ___ for v in w]
print(p)
```

목표: 확률·선택의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q39_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q40 ·로짓에 temperature 적용 (디버깅)

아래 코드는 T의 효과를 구현하려했지만 사용하지 않았다. exp(x)는 자연 지수 함수다. 로짓을 T로 나누도록 최소 수정하고, T =1과 T =2중 어느 쪽의 최댓값 확률이 작은지 설명하라.

```python
from math import exp
z = [0.0, 2.0]
T = 2.0
w = [exp(v) for v in z]
p = [v / sum(w) for v in w]
```

목표: 확률·선택의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q40_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q41 ·동률일 때 greedy (출력·해석)

코드 출력을 구하고 동률 정책을 설명하라. range(n)은 0..n −1이며 >는 엄격하게 큰 경우 만 참이다.

```python
p = [0.4, 0.4, 0.2]
best = 0
for i in range(1, len(p)):
    if p[i] > p[best]:
        best = i
print(best)
```

목표: 확률·선택의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q41_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q42 ·top-p 종료 조건 (빈칸)

분포는 이미 내림 차 순이다. 누적 값이 threshold 이상이면 멈추도록 빈칸을 채우고 keep을 구하라. 경계 후보를 먼저 추가한다.

```python
p = [0.5, 0.25, 0.15, 0.10]
threshold = 0.7
keep = []
cumulative = 0
for i in range(len(p)):
    keep.append(i)
    cumulative += p[i]
    if ___:
        break
print(keep)
```

목표: 확률·선택의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q42_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q43 ·최댓값 빼기와 넘침 (디버깅)

아래 코드는 큰 로짓의 지수에서 OverflowError가 날 수 있다. 모든 점수에서 공통 최댓 값을 빼서 수정하라. exp는 자연 지수 함수이고 결과는 소수 넷째 자리까지 제시하라.

```python
from math import exp
z = [1000.0, 1001.0]
w = [exp(v) for v in z]
p = [v / sum(w) for v in w]
```

목표: 확률·선택의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q43_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q44 ·필터 뒤 분포 읽기 (출력·해석)

아래 출력 두 개를 구하고 mass와 q가 서로 다른 대상을 나타내는 이유를 설명하라. kept는 미리 선택된 후보 인덱스다.

```python
p = [0.4, 0.35, 0.25]
kept = [0, 2]
mass = sum(p[i] for i in kept)
q = [p[i] / mass if i in kept else 0 for i in range(len(p))]
print(round(mass, 2))
print([round(v, 4) for v in q])
```

목표: 확률·선택의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q44_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q53 ·입력과 target 슬라이스 (빈칸)

빈칸 두 곳을 채워 한 칸 이동한 다음 토큰 학습 쌍을 만들어라. a[:-1]은 마지막 제외, a[1:]은 첫 항목 제외다.

```python
seq = [0, 4, 2, 9]
x = ___
y = ___
print(list(zip(x, y)))
```

목표: 생성·학습·평가의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q53_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q54 ·문맥을 덮어쓴 생성 (디버깅)

규칙 next _id[현재 마지막 ID]로 다음 ID를 얻는다. 초기 문맥[0], 두 번 생성 후 전체 문맥을 보존하려는 데 코드가 덮어 쓴다. 한 줄을 수정하고 결과를 구하라.

```python
next_id = {0: 1, 1: 2, 2: 0}
context = [0]
for _ in range(2):
    token_id = next_id[context[-1]]
    context = [token_id]
print(context)
```

목표: 생성·학습·평가의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q54_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q55 ·평가 비교의 분모 (빈칸)

빈칸을 채워 정확도를 계산하라. 정답·예측은 같은 순서이며 길이가 같다. True는 조건 충족을 뜻한다.

```python
truth = [1, 0, 1, 1]
pred = [1, 1, 0, 1]
correct = 0
for i in range(len(truth)):
    if pred[i] == truth[i]:
        correct += 1
accuracy = correct / ___
print(accuracy)
```

목표: 생성·학습·평가의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q55_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


### Q56 ·종료 토큰 처리 (출력·해석)

아래 코드의 출력과 append가 실행되는 횟수를 구하라. "끝"은 화면에 보이지 않게 종료하는 특수 토큰이라고 정의한다.

```python
chosen = ["나", "는", "끝", "학생"]
output = []
for item in chosen:
    if item == "끝":
        break
    output.append(item)
print(output)
```

목표: 생성·학습·평가의 원리와 코드를 연결합니다. 힌트: 변수의 현재 값과 반복의 범위를 한 줄씩 추적하세요. 자가 점검: 분모·행/열·처리 순서가 문제의 조건과 같은지 확인하세요.


In [ ]:
q56_my_work = ''  # 이 문자열에 풀이 이유를 적고 새 코드 셀에서 직접 완성하세요.


## 5. 독립 실행 검사

이 검사는 설명 예제와 공통 모듈의 수치·마스크·callback을 확인합니다. 연습 문항의 정답을 공개하지 않으며,본인의 풀이를 자동 채점하지 않습니다.


In [ ]:
assert math.isclose(sum(softmax_scores([4,3,1])), 1)
assert softmax_scores([10000,10000]) == [0.5,0.5]
filtered = filter_probs([.55,.25,.20], top_p=.7)
assert filtered['kept'] == [0,1] and filtered['final'][2] == 0
assert math.isclose(sum(filtered['final']),1)
masked = attention_row([0,0], [[1,0],[0,1]], [[2,0],[4,6]],0,True)
assert masked['weights'] == [1.,0.] and masked['output']==[2.,0.]
full = attention_row([0,0], [[1,0],[0,1]], [[2,0],[4,6]],1,True)
assert full['output']==[3.,3.]
assert bigram_counts('xyxz') == {'x':{'y':1,'z':1},'y':{'x':1}}
assert generate_one('x',bigram_counts('xyxz'),11,True)['context']=='xy'
for callback,args in [(probability_view,(1.,2,.9)),(attention_view,(1,True))]:
    values=callback(*args)
    assert values[0] and values[1] and values[2] is not None
    plt.close(values[2])
assert len(generation_view('a',7,True)[0])==2
print('W07B-A RUNTIME CHECK PASSED')


마지막 웹 앱 활동은[Colab B](https://colab.research.google.com/github/lunalab-ai/genAI/blob/2026-fall-w07b/notebooks/student/w07b_review_b.ipynb)에서 진행합니다. B는A의런타임 변수를 사용하지 않고 독립적으로 시작합니다.
